# camel beauty assessor — preprocessing

converts the merged annotation set into the datasets the training notebook consumes. the split is
drawn once, and every artefact below inherits it, so the same 28 test images stay unseen across
every experiment the project reports.

| dataset | contents | isolates |
|---|---|---|
| `camel-rgb` | 196 images in original colour | the reference input |
| `camel-gray` | the same 196 images, desaturated | the contribution of colour |
| `camel-rgb-aug` | 610 images, training split turned four ways | whether rotation recovers anything at this sample size |
| `camel-gray-aug` | as above, desaturated | the two factors held against each other |

resizing is not performed. ultralytics letterboxes at load time from the full-resolution source, so
input resolution stays a training parameter instead of a property baked into the files.

## 1. setup

### 1.1 imports

In [ ]:
import os
import csv
import shutil
import random
import collections

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageChops, ImageEnhance
import imagehash

### 1.2 paths

`camel-merged-v1` is the output of `dataset_eda.ipynb` and is treated as read-only. the four derived
datasets are rebuilt in full on every execution and each build removes its target directory first,
so a file left by an earlier run cannot survive into a later one and pad the counts.

In [ ]:
VERSION = "v2"

src = f"dataset/{VERSION}/merged"
out_rgb = f"dataset/{VERSION}/split/rgb"
out_gray = f"dataset/{VERSION}/split/gray"
out_rgb_aug = f"dataset/{VERSION}/augmented/rot90-rgb"
out_gray_aug = f"dataset/{VERSION}/augmented/rot90-gray"

### 1.3 display options

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.expand_frame_repr", False)

### 1.4 config

In [ ]:
seed = 42
ratios = {"train": 0.70, "val": 0.15, "test": 0.15}
rotations = [0, 90, 180, 270]

canon = ["Camel", "High_withers", "Large-head", "Large-lips", "Large-nose",
         "Large_hump", "Long-legs", "Long-neck", "Wide_body"]
traits = [c for c in canon if c != "Camel"]

assert abs(sum(ratios.values()) - 1) < 1e-9

## 2. helpers

In [ ]:
def read_manifest(folder):
    with open(os.path.join(folder, "manifest.csv"), encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def label_path(folder, image_name):
    return os.path.join(folder, "labels", os.path.splitext(image_name)[0] + ".txt")


def read_label(folder, image_name):
    with open(label_path(folder, image_name), encoding="utf-8") as fh:
        return [l.split() for l in fh if l.strip()]


def classes_in(folder, image_name):
    return [canon[int(p[0])] for p in read_label(folder, image_name)]

## 3. input check

reports the state of the merged set and modifies nothing.

In [ ]:
mf = pd.DataFrame(read_manifest(src))
images = sorted(os.listdir(os.path.join(src, "images")))

print(f"images in folder     {len(images)}")
print(f"rows in manifest     {len(mf)}")
print(f"manifest matches     {sorted(mf.new_name) == images}")

missing = [i for i in images if not os.path.exists(label_path(src, i))]
print(f"images with no label {len(missing)}")

lines = sum(len(read_label(src, i)) for i in images)
print(f"annotation lines     {lines}")
print()
print(mf.bucket.value_counts().sort_index().to_string())

In [ ]:
hashes = {i: imagehash.phash(Image.open(os.path.join(src, "images", i)).convert("RGB"))
          for i in images}

near = []
keys = list(hashes)
for a in range(len(keys)):
    for b in range(a + 1, len(keys)):
        d = hashes[keys[a]] - hashes[keys[b]]
        if d <= 8:
            near.append((d, keys[a], keys[b]))

print(f"near-duplicate pairs: {len(near)}")
if near:
    for d, a, b in sorted(near)[:10]:
        print(f"   d={d}  {a}  <->  {b}")


## 4. split

partitioned 70/15/15 and stratified on `group`, so the beautiful and negative sets hold the same
proportion in all three splits.

In [ ]:
random.seed(seed)

groups = collections.defaultdict(list)
for r in read_manifest(src):
    groups[r["group"]].append(r["new_name"])

split_of = {}
for group, names in sorted(groups.items()):
    names = sorted(names)
    random.shuffle(names)
    n_train = round(len(names) * ratios["train"])
    n_val = round(len(names) * ratios["val"])
    for i, name in enumerate(names):
        if i < n_train:
            split_of[name] = "train"
        elif i < n_train + n_val:
            split_of[name] = "val"
        else:
            split_of[name] = "test"

split_df = pd.DataFrame([{"image": k, "split": v} for k, v in sorted(split_of.items())])
split_df["group"] = split_df.image.str.extract(r"^(BEAUTY|NEGATIVE)")

pd.crosstab(split_df.group, split_df.split).reindex(columns=["train", "val", "test"])

In [ ]:
split_df["batch"] = split_df.image.str.extract(r"^[A-Z]+_([AB])")
pd.crosstab(split_df.batch, split_df.split).reindex(columns=["train", "val", "test"])

In [ ]:
rows = []
for name, sp in split_of.items():
    for c in classes_in(src, name):
        rows.append({"split": sp, "cls": c})

per_class = (pd.DataFrame(rows).groupby(["cls", "split"]).size()
             .unstack(fill_value=0)
             .reindex(index=canon, columns=["train", "val", "test"], fill_value=0))
per_class["total"] = per_class.sum(axis=1)
per_class

In [ ]:
thin = per_class[(per_class.test < 5) | (per_class.val < 5)]
if len(thin):
    print("classes with fewer than 5 boxes in val or test:")
    print(thin[["train", "val", "test"]].to_string())
    print()
    print("metrics on these will be noisy - a single missed box moves recall a lot.")
    print("worth considering k-fold cross validation instead of one fixed split.")
else:
    print("every class has at least 5 boxes in val and test")

## 5. export

In [ ]:
def export(dst, grayscale):
    shutil.rmtree(dst, ignore_errors=True)
    for sp in ratios:
        os.makedirs(os.path.join(dst, sp, "images"))
        os.makedirs(os.path.join(dst, sp, "labels"))

    for name, sp in sorted(split_of.items()):
        img = Image.open(os.path.join(src, "images", name))
        if grayscale:
            # back to 3 channels: yolo expects rgb input even when it is all grey
            img = img.convert("L").convert("RGB")
        else:
            img = img.convert("RGB")
        img.save(os.path.join(dst, sp, "images", name))
        shutil.copy2(label_path(src, name),
                     os.path.join(dst, sp, "labels", os.path.splitext(name)[0] + ".txt"))

    with open(os.path.join(dst, "data.yaml"), "w", encoding="utf-8") as fh:
        fh.write("path: .\ntrain: train/images\nval: val/images\ntest: test/images\n\n")
        fh.write(f"nc: {len(canon)}\n")
        fh.write("names: [" + ", ".join(f"'{c}'" for c in canon) + "]\n")
    return dst


export(out_rgb, grayscale=False)
export(out_gray, grayscale=True)

for d in (out_rgb, out_gray):
    counts = {sp: len(os.listdir(os.path.join(d, sp, "images"))) for sp in ratios}
    print(f"{d:<22} {counts}   total {sum(counts.values())}")

## 6. verification

In [ ]:
def collect(dst):
    out = {}
    for sp in ratios:
        for f in os.listdir(os.path.join(dst, sp, "images")):
            out.setdefault(f, []).append(sp)
    return out


checks = {}

placed = collect(out_rgb)
checks["every image placed exactly once"] = all(len(v) == 1 for v in placed.values())
checks["no image lost"] = len(placed) == len(images)

rgb_files = collect(out_rgb)
gray_files = collect(out_gray)
checks["rgb and gray have the same split"] = rgb_files == gray_files

same_labels = True
total_lines = 0
for sp in ratios:
    for f in os.listdir(os.path.join(out_rgb, sp, "labels")):
        a = open(os.path.join(out_rgb, sp, "labels", f), encoding="utf-8").read()
        b = open(os.path.join(out_gray, sp, "labels", f), encoding="utf-8").read()
        c = open(os.path.join(src, "labels", f), encoding="utf-8").read()
        total_lines += len([l for l in a.splitlines() if l.strip()])
        if not (a == b == c):
            same_labels = False
checks["labels identical to source"] = same_labels
checks["annotation lines preserved"] = total_lines == lines

for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(checks.values()), "a verification check failed"

In [ ]:
def is_grayscale(path):
    r, g, b = Image.open(path).convert("RGB").split()
    return (ImageChops.difference(r, g).getbbox() is None
            and ImageChops.difference(g, b).getbbox() is None)


sample_rgb = os.path.join(out_rgb, "train", "images", sorted(os.listdir(os.path.join(out_rgb, "train", "images")))[0])
sample_gray = sample_rgb.replace(out_rgb, out_gray)

print(f"rgb copy  is grayscale: {is_grayscale(sample_rgb)}   (expected False)")
print(f"gray copy is grayscale: {is_grayscale(sample_gray)}   (expected True)")
print()
print(f"disk: rgb {sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(out_rgb) for f in fs) / 1e6:.0f} mb")
print(f"      gray {sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(out_gray) for f in fs) / 1e6:.0f} mb")

## 7. augmentation

ultralytics augments at load time and on the training split alone, drawing a fresh transformation
each epoch and writing nothing to disk. that already covers mirroring, hue, saturation, brightness,
scale, translation and mosaic, so reproducing any of them here would reach the same image by two
routes: the dataset would grow without carrying more information.

### 7.1 transform allocation

In [ ]:
pd.DataFrame([
    ("rotate 90 / 180 / 270", "here, on disk", "yolo's degrees is 0, nothing else does this. 4x the files and 4x the instances."),
    ("mirror left to right", "training", "yolo's fliplr is 0.5. doing it here too is the same picture twice."),
    ("brightness, contrast, colour", "training", "yolo's hsv_v, hsv_s, hsv_h. covers a night runway against full daylight."),
    ("zoom and shift", "training", "yolo's scale 0.5 and translate 0.1."),
    ("four photos stitched into one", "training", "yolo's mosaic 1.0."),
    ("upside-down flip", "nowhere", "yolo's fliplr on top of our 180 turn lands on it exactly. a third route to the same picture."),
    ("blur", "nowhere", "the median lips box is about 30px at 640. blur removes the detail that class runs on."),
], columns=["transform", "applied", "why"])

### 7.2 build

In [ ]:
def rotate_point(x, y, deg):
    if deg == 90:
        return y, 1 - x
    if deg == 180:
        return 1 - x, 1 - y
    if deg == 270:
        return 1 - y, x
    return x, y


def rotate_line(parts, deg):
    vals = [float(v) for v in parts[1:]]
    if len(vals) == 4:
        xc, yc, w, h = vals
        xc, yc = rotate_point(xc, yc, deg)
        # a quarter turn swaps which side is width and which is height
        vals = [xc, yc, h, w] if deg in (90, 270) else [xc, yc, w, h]
    else:
        vals = [v for x, y in zip(vals[0::2], vals[1::2]) for v in rotate_point(x, y, deg)]
    return parts[0] + " " + " ".join(f"{min(max(v, 0.0), 1.0):.6f}" for v in vals)


def augment(source, dst):
    shutil.rmtree(dst, ignore_errors=True)
    for sp in ratios:
        os.makedirs(os.path.join(dst, sp, "images"))
        os.makedirs(os.path.join(dst, sp, "labels"))

    made = collections.Counter()
    for sp in ratios:
        turns = rotations if sp == "train" else [0]
        for name in sorted(os.listdir(os.path.join(source, sp, "images"))):
            stem, ext = os.path.splitext(name)
            for deg in turns:
                tag = stem if deg == 0 else f"{stem}_r{deg}"
                img_out = os.path.join(dst, sp, "images", tag + ext)
                lbl_out = os.path.join(dst, sp, "labels", tag + ".txt")

                if deg == 0:
                    shutil.copy2(os.path.join(source, sp, "images", name), img_out)
                    shutil.copy2(label_path(os.path.join(source, sp), name), lbl_out)
                else:
                    Image.open(os.path.join(source, sp, "images", name)).rotate(deg, expand=True).save(img_out, quality=95)
                    rows = read_label(os.path.join(source, sp), name)
                    with open(lbl_out, "w", encoding="utf-8") as fh:
                        fh.writelines(rotate_line(p, deg) + "\n" for p in rows)
                made[sp] += 1

    shutil.copy2(os.path.join(source, "data.yaml"), os.path.join(dst, "data.yaml"))
    return made


for source, dst in ((out_rgb, out_rgb_aug), (out_gray, out_gray_aug)):
    made = augment(source, dst)
    print(f"{dst:<26} {dict(made)}   total {sum(made.values())}")

### 7.3 verification

In [ ]:
def count_lines(root, sp):
    d = os.path.join(root, sp, "labels")
    return sum(1 for f in os.listdir(d)
               for l in open(os.path.join(d, f), encoding="utf-8") if l.strip())


aug_checks = {}
n_before = {sp: len(os.listdir(os.path.join(out_rgb, sp, "images"))) for sp in ratios}
n_after = {sp: len(os.listdir(os.path.join(out_rgb_aug, sp, "images"))) for sp in ratios}

aug_checks["train grew by exactly 4x"] = n_after["train"] == n_before["train"] * len(rotations)
aug_checks["train instances grew by exactly 4x"] = (
    count_lines(out_rgb_aug, "train") == count_lines(out_rgb, "train") * len(rotations))
aug_checks["val and test untouched"] = all(n_after[sp] == n_before[sp] for sp in ("val", "test"))
aug_checks["no rotated copy reached val or test"] = not any(
    os.path.splitext(f)[0].endswith(("_r90", "_r180", "_r270"))
    for sp in ("val", "test") for f in os.listdir(os.path.join(out_rgb_aug, sp, "images")))

bad_range, orphan = [], []
for sp in ratios:
    for f in sorted(os.listdir(os.path.join(out_rgb_aug, sp, "images"))):
        stem, _ = os.path.splitext(f)
        if not os.path.exists(os.path.join(out_rgb_aug, sp, "labels", stem + ".txt")):
            orphan.append(f)
            continue
        for parts in read_label(os.path.join(out_rgb_aug, sp), f):
            if any(not 0 <= float(v) <= 1 for v in parts[1:]):
                bad_range.append(f)

aug_checks["every image has a label"] = not orphan
aug_checks["every coordinate inside [0, 1]"] = not bad_range

turned = [f for f in os.listdir(os.path.join(out_rgb_aug, "train", "images"))
          if os.path.splitext(f)[0].endswith("_r90")]
swapped = True
for f in turned:
    stem, ext = os.path.splitext(f)
    base = stem[:-4] + ext
    w, h = Image.open(os.path.join(out_rgb_aug, "train", "images", f)).size
    sw, sh = Image.open(os.path.join(out_rgb, "train", "images", base)).size
    if (w, h) != (sh, sw):
        swapped = False
aug_checks["a quarter turn swapped width and height"] = swapped

sample = sorted(os.listdir(os.path.join(out_rgb, "train", "images")))[0]
start = read_label(os.path.join(out_rgb, "train"), sample)
round_trip = [list(p) for p in start]
for _ in range(4):
    round_trip = [rotate_line(p, 90).split() for p in round_trip]
aug_checks["four turns of 90 return the original"] = all(
    abs(float(a) - float(b)) < 1e-4
    for r0, r4 in zip(start, round_trip) for a, b in zip(r0[1:], r4[1:]))

aug_checks["rgb and gray aug hold the same images"] = all(
    sorted(os.listdir(os.path.join(out_rgb_aug, sp, "images")))
    == sorted(os.listdir(os.path.join(out_gray_aug, sp, "images"))) for sp in ratios)
aug_checks["the gray copy is still gray after rotating"] = is_grayscale(
    os.path.join(out_gray_aug, "train", "images", turned[0]))

for k, v in aug_checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
assert all(aug_checks.values()), "an augmentation check failed"

### 7.4 instance count

instances rather than images is the figure a detector is measured against, as one photograph carries
between 1 and 17 annotated objects in this set.

In [ ]:
def tally(root):
    rows = []
    for sp in ["train", "val", "test"]:
        cc = collections.Counter()
        d = os.path.join(root, sp, "labels")
        for f in os.listdir(d):
            for l in open(os.path.join(d, f), encoding="utf-8"):
                if l.strip():
                    cc[canon[int(l.split()[0])]] += 1
        rows.append({"split": sp,
                     "images": len(os.listdir(os.path.join(root, sp, "images"))),
                     "instances": sum(cc.values()), **cc})
    return pd.DataFrame(rows).set_index("split").fillna(0).astype(int)


before, after = tally(out_rgb), tally(out_rgb_aug)

print(f"images      {before.images.sum():>5}  ->  {after.images.sum():>5}")
print(f"instances   {before.instances.sum():>5}  ->  {after.instances.sum():>5}")
print()
print("train only. val and test are the same real photos as before, on purpose:")
print("a score is only worth reading if it was measured on images nothing generated.")

pd.concat({"before": before[["images", "instances"]], "after": after[["images", "instances"]]}, axis=1)

In [ ]:
per_class_aug = pd.DataFrame({"before": before.loc["train"], "after": after.loc["train"]}).reindex(canon)
per_class_aug["x"] = (per_class_aug.after / per_class_aug.before).round(1)
per_class_aug

### 7.5 geometry check

In [ ]:
def to_xyxy(parts, w, h):
    v = [float(x) for x in parts[1:]]
    if len(v) == 4:
        xc, yc, bw, bh = v
        x0, y0, x1, y1 = xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2
    else:
        # polygons are drawn as their enclosing rectangle here, this figure is about orientation
        xs, ys = v[0::2], v[1::2]
        x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
    return x0 * w, y0 * h, x1 * w, y1 * h


busiest = max(os.listdir(os.path.join(out_rgb, "train", "images")),
              key=lambda f: len(read_label(os.path.join(out_rgb, "train"), f)))
stem, ext = os.path.splitext(busiest)

fig, axes = plt.subplots(1, len(rotations), figsize=(4.2 * len(rotations), 4.6))
for ax, deg in zip(axes, rotations):
    name = stem + ext if deg == 0 else f"{stem}_r{deg}{ext}"
    img = Image.open(os.path.join(out_rgb_aug, "train", "images", name))
    ax.imshow(img)
    for parts in read_label(os.path.join(out_rgb_aug, "train"), name):
        x0, y0, x1, y1 = to_xyxy(parts, *img.size)
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1.3,
                                   edgecolor=plt.cm.tab10.colors[int(parts[0]) % 10]))
    ax.set_title(f"{deg} degrees", fontsize=9)
    ax.axis("off")
    ax.grid(False)

fig.suptitle(f"{busiest}, one image and its four turns - the boxes follow")
plt.tight_layout()
plt.show()

## 8. output

In [ ]:
summary = []
for d, label in ((out_rgb, "rgb"), (out_gray, "gray"),
                 (out_rgb_aug, "rgb aug"), (out_gray_aug, "gray aug")):
    for sp in ["train", "val", "test"]:
        n_img = len(os.listdir(os.path.join(d, sp, "images")))
        n_box = sum(len(open(os.path.join(d, sp, "labels", f), encoding="utf-8").readlines())
                    for f in os.listdir(os.path.join(d, sp, "labels")))
        summary.append({"dataset": label, "split": sp, "images": n_img, "annotations": n_box})

pd.DataFrame(summary).pivot(index="split", columns="dataset", values=["images", "annotations"]).reindex(["train", "val", "test"])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
per_class[["train", "val", "test"]].plot(kind="bar", stacked=True, ax=ax,
                                         color=["#4c72b0", "#dd8452", "#55a868"])
ax.set_ylabel("annotation boxes")
ax.set_xlabel("")
ax.set_title("class balance across the splits")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()